In [25]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [26]:
### Load the trained model, scaler pickle,onehot
model=load_model('model.keras')

## load the encoder and scaler
with open('encode_hot.pkl','rb') as file:
    encode_hot = pickle.load(file)

with open('encode_label.pkl', 'rb') as file:
    encod_label = pickle.load(file)

with open('scaled.pkl', 'rb') as file:
    scaled = pickle.load(file)

Example input data

In [27]:

input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 78,
    'Tenure': 4,
    'Balance': 600000,
    'NumOfProducts': 0,
    'HasCrCard': 0,
    'IsActiveMember': 1,
    'EstimatedSalary': 5000000
}

One-hot encode 'Geography'

In [28]:

geo_encoded = encode_hot.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=encode_hot.get_feature_names_out(['Geography']))
geo_encoded_df


c:\dl project\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [29]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,78,4,600000,0,0,1,5000000


In [30]:
input_df['Gender']=encod_label.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,78,4,600000,0,0,1,5000000


In [31]:
input_df.drop(["Geography"], axis=1, inplace=True)
input_df=pd.concat([input_df, geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,78,4,600000,0,0,1,5000000,1.0,0.0,0.0


Scaling the input data

In [32]:
input_scaled=scaled.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  3.71754617, -0.3483691 ,  8.38812313,
        -2.6418115 , -1.54035103,  0.97481699, 85.1871858 ,  1.00150113,
        -0.57946723, -0.57638802]])

In [33]:
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step


array([[0.9961484]], dtype=float32)

In [34]:
prediction_proba = prediction[0][0]

In [35]:
prediction_proba

np.float32(0.9961484)

In [36]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is likely to churn.
